# 04 — Configurable domains, root selection, and pseudo-time
Compare 7-domain and sensitivity configurations, explicitly select the root, and compute root-relative pseudo-time.

In [ ]:
!pip install -q --no-cache-dir --force-reinstall git+https://github.com/YoshifumiMiyagi/Fontan.git


In [ ]:
import pandas as pd
from fontan_manifold import FontanManifold

DATA_PATH = '/content/drive/MyDrive/Fontan/your_data.csv'
raw = pd.read_csv(DATA_PATH)
df = FontanManifold.prepare_dataframe(raw)


## A. Main 7-domain model

In [ ]:
model7 = FontanManifold(domains='7domain', n_neighbors=50, n_dm=5, id_col='subj_id')
model7.fit(df)
print('Reference N =', len(model7.reference_df_))
display(model7.domain_summary())
display(model7.self_validate(n_dm=3))


## B. Root option 1: favorable clinical-burden score
Positive direction = higher is worse; negative direction = higher is favorable. The root is the reference subject with the lowest standardized burden score.

In [ ]:
root = model7.set_root(
    method='score',
    score_directions={
        'log_BNP': +1,
        'echoedv': +1,
        'echomass': +1,
        'echoef': -1,
        'pp_peakvo2': -1,
        'pp_vat': -1,
    },
    dms=('DM1','DM2','DM3'),
)
print(root)

reference_pt = model7.compute_pseudotime(dms=('DM1','DM2','DM3'))
display(reference_pt.head())


## C. Root sensitivity options
Use these as sensitivity analyses rather than silently changing the primary root.

In [ ]:
# Geometric endpoint
print(model7.set_root(method='dm1_min', dms=('DM1','DM2','DM3')))
pt_dm1min = model7.compute_pseudotime(dms=('DM1','DM2','DM3'))

# Explicit clinically selected reference subject
# print(model7.set_root(method='subject_id', subject_id=123, dms=('DM1','DM2','DM3')))
# pt_subject = model7.compute_pseudotime(dms=('DM1','DM2','DM3'))


## D. Domain sensitivity analyses

In [ ]:
models = {}
for preset in ['7domain','6domain_no_biomarker','6domain_no_exercise','5domain_echo_core']:
    m = FontanManifold(domains=preset, n_neighbors=50, n_dm=5, id_col='subj_id')
    m.fit(df)
    models[preset] = m
    print(preset, 'Reference N =', len(m.reference_df_))
    display(m.self_validate(n_dm=3))


## E. Custom domain combination
Any subset of the seven named domains can be supplied explicitly.

In [ ]:
custom = FontanManifold(
    domains=['Exercise','Remodeling','Function','Diastolic','Valve','Biomarker'],
    n_neighbors=50,
    n_dm=5,
)
custom.fit(df)
display(custom.domain_summary())


## F. MI projection followed by pseudo-time
The root is defined in the fixed reference; the same root coordinate is then used for projected subjects.

In [ ]:
# Restore the prespecified primary root before MI projection
model7.set_root(
    method='score',
    score_directions={'log_BNP':+1,'echoedv':+1,'echomass':+1,'echoef':-1,'pp_peakvo2':-1,'pp_vat':-1},
    dms=('DM1','DM2','DM3'),
)

projection_long, diagnostics = model7.project_multiple_imputation(
    df, n_imputations=20, seeds=range(1001,1021), n_dm=3
)

pt_long = []
for mi, g in projection_long.groupby('MI'):
    tmp = model7.compute_pseudotime(g, dms=('DM1','DM2','DM3'))
    tmp['MI'] = mi
    pt_long.append(tmp)
pt_long = pd.concat(pt_long, ignore_index=True)

pt_summary = pt_long.groupby('subj_id')['pseudo_time'].agg(['mean','std']).reset_index()
display(pt_summary.head())
